# CS383: Data Science and Machine Learning
## Lecture 7 — Linear Regression

*Dr. Thitima Srivatanakul*

### Guiding question
**How does a model turn a scatter of points into a single "best" line — and how do you tell the
difference between a model that found something real and one that's honestly telling you it found
nothing?**

### Learning objectives
By the end of this lecture, you should be able to:

- derive the least-squares solution for simple linear regression, and confirm it matches what
  scikit-learn's `LinearRegression` fits;
- fit and correctly interpret a multiple linear regression model's coefficients;
- compute and explain MAE, MSE, RMSE, and R² — including what each one does and doesn't tell you;
- read a residual plot, and recognize that "well-behaved residuals" and "a good model" are two
  different questions;
- report a weak or near-zero R² honestly, as real information, rather than a failure to hide.

---

### Where this fits
Lecture 6 already prepped two datasets for exactly this moment: restaurant inspection features (encoded,
scaled) and NYC 311's `resolution_time_hours`, explicitly set aside as "a real regression target in
Week 7." Today we finally build the model — starting from the actual math, not just an API call, so
`LinearRegression()` stops being a black box. Fair warning going in: real civic data of the kind this
whole course uses is often *genuinely hard to predict* from the columns you happen to have. This lecture
doesn't hide that — it teaches you how to notice it, and what to do about it.

---

## Part 1 — Simple Linear Regression, From Scratch

Before letting scikit-learn do it for us, let's see exactly what "fitting a line" means.

### Setup — NYC restaurant inspections

Same source as Lecture 6, rebuilt at the row-level (one row per violation citation) rather than
collapsed into groups — we want `is_critical` and `grade` to describe an actual real record, not an
artifact of how rows happened to get grouped.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score", "grade"]).reset_index(drop=True)

    # One row here is one violation citation, not one full inspection -- be careful not to call this
    # "violations per inspection," since a single inspection can contribute more than one row.
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)
    inspections_df["grade_ord"] = inspections_df["grade"].map({"A": 0, "B": 1, "C": 2})
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room. Deliberately built with
    # score unrelated to is_critical/grade -- that's what the real shared snapshot actually shows too.
    rng = np.random.default_rng(383)
    n = 4000
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza", "Japanese"]

    is_critical = rng.integers(0, 2, size=n)
    grade = rng.choice(["A", "B", "C"], size=n, p=[0.6, 0.25, 0.15])
    score = rng.integers(0, 71, size=n)

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_description": rng.choice(cuisines_clean, size=n),
        "score": score,
        "grade": grade,
        "critical_flag": np.where(is_critical == 1, "Critical", "Not Critical"),
        "is_critical": is_critical,
    })
    inspections_df["grade_ord"] = inspections_df["grade"].map({"A": 0, "B": 1, "C": 2})
    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(inspections_df):,} violation records")
inspections_df[["boro", "cuisine_description", "score", "grade", "is_critical"]].head()

In [ ]:
rng_jitter = np.random.default_rng(0)
x_jittered = inspections_df["is_critical"] + rng_jitter.uniform(-0.08, 0.08, size=len(inspections_df))

plt.scatter(x_jittered, inspections_df["score"], alpha=0.15, s=10)
plt.xlabel("Critical violation? (0 = No, 1 = Yes, jittered for visibility)")
plt.ylabel("Score")
plt.title("Score vs. Critical-Violation Flag")
plt.show()

`is_critical` is 0/1, so without jitter every point would stack into two vertical lines — adding a
little random horizontal noise *just for plotting* spreads them out enough to see the density. Look
closely: the two clouds barely differ. That's worth taking seriously, not glossing over — we'll quantify
exactly how little they differ starting in Part 3.

### The model: one line through the cloud

A simple linear regression model says the relationship between one input $x$ and output $y$ can be
approximated by a straight line:

$$\hat{y} = b_0 + b_1 x$$

- $\hat{y}$ ("y-hat") is the model's *predicted* score — not the actual one.
- $b_0$ is the **intercept**: the predicted score when `is_critical` is exactly 0.
- $b_1$ is the **slope**: how much the predicted score changes when `is_critical` goes from 0 to 1.

Every choice of $b_0$ and $b_1$ draws a different line. "Fitting" the model means picking the *one* line
that fits this data best — regardless of whether that best line turns out to be a good line.

### Defining "best": residuals and the cost function

For any candidate line, the **residual** for a single row is how far off that line's prediction was:

$$e_i = y_i - \hat{y}_i$$

A good line has small residuals across the board. To turn "small residuals across the board" into a
single number we can minimize, linear regression uses the **sum of squared residuals**:

$$SSR = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Why *squared*, instead of just summing the residuals directly? Two reasons: squaring makes every term
positive, so overestimates and underestimates can't cancel out — and it also gives a smooth function we
can minimize with calculus (next). It has a side effect worth remembering, too: squaring makes a residual
of 10 count 100x as much as a residual of 1 — large misses are punished disproportionately.

### Deriving the least-squares solution

We want the $b_0$ and $b_1$ that make $SSR$ as small as possible. Calculus tells us: at a minimum, the
*partial derivatives* of $SSR$ with respect to $b_0$ and $b_1$ are both zero. Working through that (you
don't need to reproduce this derivation yourself, but you should be able to follow it) gives two clean,
closed-form formulas:

$$b_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2} \qquad b_0 = \bar{y} - b_1 \bar{x}$$

In words: the slope is the covariance between $x$ and $y$, divided by the variance of $x$. The intercept
just makes sure the line passes through the point $(\bar{x}, \bar{y})$ — the "center of mass" of the
data. Notice this formula has no opinion about whether $x$ and $y$ are actually related — if the
covariance in the numerator is near zero, it will honestly hand back a slope near zero.

In [ ]:
x = inspections_df["is_critical"].values
y = inspections_df["score"].values

x_bar = x.__________()
y_bar = y.__________()

b1_byhand = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
b0_byhand = y_bar - b1_byhand * x_bar

print(f"By-hand intercept (b0): {b0_byhand:.4f}")
print(f"By-hand slope     (b1): {b1_byhand:.4f}")

In [ ]:
simple_model = __________()
simple_model.fit(inspections_df[["is_critical"]], inspections_df["score"])

print(f"scikit-learn intercept: {simple_model.intercept_:.4f}")
print(f"scikit-learn slope:     {simple_model.coef_[0]:.4f}")

Same numbers (up to rounding). `LinearRegression()` isn't doing anything mysterious — it's solving the
exact same minimization problem we just derived by hand, and it's just as honest about a near-zero
slope as our by-hand version was. Every regression model in this course builds on this same idea: define
a cost function, find the parameters that minimize it — whatever those parameters turn out to be.

In [ ]:
x_range = np.__________(-0.1, 1.1, 100)
y_line = b0_byhand + b1_byhand * x_range

plt.scatter(x_jittered, y, alpha=0.15, s=10, label="Actual inspections")
plt.plot(x_range, y_line, color="#D98C3F", linewidth=2, label="Fitted line")
plt.xlabel("Critical violation? (jittered)")
plt.ylabel("Score")
plt.title("The Least-Squares Line")
plt.legend()
plt.show()

The fitted line is nearly flat. That's not a plotting mistake — it's the honest least-squares answer for
this particular $x$ and $y$. A flat line is still a real fitted model; it's just a model whose best
prediction barely changes no matter what you feed it.

---

## Part 2 — Multiple Linear Regression

Real problems rarely stop at one predictor. `grade` (ordinal-encoded, from Lecture 6) is sitting right
there too — let's use both `is_critical` and `grade_ord` together.

### From one line to a matrix equation

With two predictors, the model becomes:

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2$$

and in general, with $p$ predictors:

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \dots + b_p x_p$$

This is much easier to write — and to solve — in matrix form. Stack a column of 1s onto your feature
matrix (for the intercept), and the whole model becomes $\hat{y} = X\beta$, where $\beta$ is the vector of
all the $b$'s. Minimizing the same sum-of-squared-residuals cost function now has a closed-form matrix
solution, the **normal equation**:

$$\beta = (X^T X)^{-1} X^T y$$

This is the direct generalization of the two formulas from Part 1 — same idea, just written for many
predictors at once.

In [ ]:
X_raw = inspections_df[["is_critical", "grade_ord"]].values
X_design = np.column_stack([np.ones(len(X_raw)), X_raw])  # add intercept column of 1s
y_vals = inspections_df["score"].values

beta_byhand = np.linalg.__________(X_design.T @ X_design) @ X_design.T @ y_vals

print("By-hand coefficients [intercept, is_critical, grade_ord]:")
print(beta_byhand.round(4))

In [ ]:
multi_model = LinearRegression()
multi_model.__________(inspections_df[["is_critical", "grade_ord"]], inspections_df["score"])

print(f"scikit-learn intercept: {multi_model.intercept_:.4f}")
print(f"scikit-learn coefficients: {multi_model.coef_.round(4)}")

Same numbers again. (In practice, scikit-learn doesn't literally invert a matrix like we just did —
matrix inversion is numerically unstable for larger problems, so it uses a more stable least-squares
solver internally. The *math it's solving* is identical either way.)

### Interpreting the coefficients

Each coefficient is a **partial effect**: holding every other feature fixed, how much does the predicted
score change for a one-unit increase in that feature? Look at the actual sizes of both coefficients above
— they should be small, close to zero. That's a real, honest finding: within this dataset, neither
whether a violation was critical, nor the letter grade on file, moves the predicted score by much once
you account for both together.

This is also a good moment for a caution that applies whether coefficients are large or tiny: a
coefficient describes the model's fitted line, not a causal claim. A large coefficient wouldn't prove
"critical violations *cause* worse scores" any more than a near-zero one proves they don't matter in
the real world — it only describes the pattern this particular line captured in this particular data.

---

## Part 3 — Evaluating a Regression Model

An $R^2$, an MAE, an RMSE — the Iris sneak peek used these informally. Here's what each one actually
measures, and when to reach for which — using the honest, weak-signal model from Part 2 evaluated
properly on a held-out test set.

In [ ]:
X_scores = inspections_df[["is_critical", "grade_ord"]]
y_scores = inspections_df["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X_scores, y_scores, test_size=__________, random_state=383
)

eval_model = LinearRegression()
eval_model.fit(X_train, y_train)
y_pred = eval_model.predict(X_test)

### Mean Absolute Error (MAE)

$$MAE = \frac{1}{n}\sum |y_i - \hat{y}_i|$$

The average size of a miss, in the original units (points, here) — the most directly interpretable
metric. It treats every error in proportion to its size, so one huge miss doesn't dominate the number.

In [ ]:
mae = __________(y_test, y_pred)
print(f"MAE: {mae:.2f} points")

### Mean Squared Error (MSE) and Root Mean Squared Error (RMSE)

$$MSE = \frac{1}{n}\sum (y_i - \hat{y}_i)^2 \qquad RMSE = \sqrt{MSE}$$

Squaring the errors (same reasoning as Part 1's cost function) makes large misses count
disproportionately more. RMSE takes the square root at the end, bringing the units back to the original
scale (points, not points²) so it's directly comparable to MAE — and $RMSE \geq MAE$ always. The gap
between them tells you something: a big gap means a few large errors are dragging the average up; a small
gap means errors are fairly uniform in size.

In [ ]:
mse = __________(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f} points")

### R² (coefficient of determination)

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

$SS_{res}$ is the same sum-of-squared-residuals from Part 1's cost function — how much error the model
actually made. $SS_{tot}$ is how much error you'd make with the simplest possible model: always
predicting the mean, $\bar{y}$. $R^2$ is the fraction of that baseline error your model *eliminated*.

- $R^2 = 1$: perfect predictions.
- $R^2 = 0$: your model does no better than just guessing the mean every time.
- $R^2 < 0$: your model does *worse* than guessing the mean — a real possibility on a bad fit or a badly
  mismatched test set, not just a theoretical edge case.

In [ ]:
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2_byhand = 1 - ss_res / ss_tot

r2_sklearn = __________(y_test, y_pred)

print(f"By-hand R²: {r2_byhand:.4f}")
print(f"sklearn R²: {r2_sklearn:.4f}")

That R² should be extremely close to zero — maybe even negative on this particular test split. Sit with
that for a second instead of rushing past it: it means `is_critical` and `grade_ord` together explain
essentially none of the variation in `score`, on data pulled straight from NYC Open Data. That's a real,
useful finding about *this* dataset, not a sign the math above was done wrong — the by-hand and sklearn
numbers matching each other (as they did in Parts 1 and 2) is what proves the math is right. Whether the
result is exciting is a completely separate question from whether it's correct.

---

## Part 4 — Checking the Model: Residual Plots

A single R² number can hide a lot — but a residual plot can also mislead you in the *other* direction if
you're not careful about what it can and can't tell you.

In [ ]:
residuals = y_test - __________

plt.scatter(y_pred, residuals, alpha=0.4, s=15)
plt.axhline(0, color="#9AA5B1", linestyle="--")
plt.xlabel("Predicted score")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residuals vs. Predicted Score")
plt.show()

What a residual plot checks for is narrow and specific: is there a *pattern* left over — a funnel shape
(heteroscedasticity, the model's errors growing or shrinking with the prediction) or a curve (a sign the
true relationship isn't linear at all)? Because this model only had two near-useless 0/1/ordinal
features to predict from, its predictions barely vary — so this plot will likely look like a few thin
vertical bands of dots, each one flat and centered on zero. That flatness is genuinely a good sign in the
narrow sense: the model isn't missing an obvious nonlinear pattern in the features you gave it.

**It does not mean the model is good.** "No leftover pattern in the residuals" and "the model explains a
lot of the outcome" are two completely different questions — the residual plot answers the first, R²
answers the second. This model can pass the residual-plot check with flying colors and still have an R²
near zero. Both things can be true about the same model at once.

In [ ]:
plt.__________(residuals, bins=30, color="#2E5C8A", edgecolor="white")
plt.xlabel("Residual")
plt.title("Distribution of Residuals")
plt.show()

Roughly symmetric and centered around 0 is what you're checking for here, and it holds up fine — but
notice the spread is wide relative to how little the model's predictions actually move. That wide,
symmetric spread of residuals *is* the near-zero R² from Part 3, seen from a different angle: the model
just isn't narrowing down the range of likely scores by much.

---

## Part 5 — A Second Real Target: NYC 311 Resolution Time

Lecture 6 already prepped this exact target — `resolution_time_hours` — and explicitly set it aside as
"a real regression target in Week 7." Let's see whether three genuinely different features
(`complaint_type`, `borough`, `hour_filed`) fare any better than Part 2's did.

### Setup — NYC 311

Same dataset and prep as Lecture 6, Part 6.

In [ ]:
import os

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_weight = np.where(days.dayofweek >= 5, 0.6, 1.0)
    day_weight = day_weight / day_weight.sum()

    day_idx = rng.choice(n_days, size=n, p=day_weight)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")

    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df[["complaint_type", "borough", "hour_filed", "resolution_time_hours"]].head()

In [ ]:
complaints_clean = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)

X_311 = complaints_clean[["complaint_type", "borough", "hour_filed"]]
y_311 = complaints_clean["resolution_time_hours"]

X_311_train, X_311_test, y_311_train, y_311_test = train_test_split(
    X_311, y_311, test_size=0.2, random_state=383
)

print("Training rows:", len(X_311_train))
print("Test rows:    ", len(X_311_test))

In [ ]:
preprocessor_311 = ColumnTransformer(transformers=[
    ("num", StandardScaler(), ["hour_filed"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["complaint_type", "borough"]),
])

X_311_train_ready = preprocessor_311.fit_transform(X_311_train)
X_311_test_ready = preprocessor_311.transform(X_311_test)

print("Training features shape:", X_311_train_ready.shape)
print("Test features shape:    ", X_311_test_ready.shape)

In [ ]:
model_311 = __________()
model_311.fit(X_311_train_ready, y_311_train)
y_311_pred = model_311.predict(X_311_test_ready)

mae_311 = mean_absolute_error(y_311_test, y_311_pred)
rmse_311 = np.sqrt(mean_squared_error(y_311_test, y_311_pred))
r2_311 = r2_score(y_311_test, y_311_pred)

print(f"MAE:  {mae_311:.2f} hours")
print(f"RMSE: {rmse_311:.2f} hours")
print(f"R²:   {r2_311:.4f}")

Look at that R² next to Part 3's. It might be a little higher, a little lower, or about the same order
of magnitude — either way, it's still close to zero, and that's the real, consistent finding of this
entire lecture: `complaint_type`, `borough`, and the hour a complaint was filed are real, legitimate
signal to try, but they're nowhere close to the full story. How long a complaint actually takes depends
on that agency's current backlog, staffing that specific day, weather, and plenty else we simply don't
have in this dataset. A model this weak is still telling you something true and useful — these columns
alone don't explain much of what drives resolution time. Reporting that plainly, instead of quietly
picking a rosier metric or not mentioning R² at all, is exactly the kind of judgment call your capstone
write-up will need to make.

In [ ]:
residuals_311 = y_311_test - __________

plt.scatter(y_311_pred, residuals_311, alpha=0.3, s=12)
plt.axhline(0, color="#9AA5B1", linestyle="--")
plt.xlabel("Predicted resolution time (hours)")
plt.ylabel("Residual")
plt.title("Residuals vs. Predicted Resolution Time")
plt.show()

Expect a lopsided, fan-shaped spread here, different from Part 4's thin flat bands — the model predicts
a fairly narrow range of resolution times, but the actual values include a long tail of complaints that
took much longer than predicted. That shape is a direct fingerprint of `resolution_time_hours` itself:
most complaints close quickly, a smaller number take a very long time, and a symmetric straight line is
a poor match for that kind of skew — a different, additional problem on top of the weak-signal issue
Part 3 already surfaced.

In [ ]:
plt.__________(y_311_train, bins=40, color="#2E5C8A", edgecolor="white")
plt.xlabel("Resolution time (hours)")
plt.title("Distribution of Resolution Time (Training Set)")
plt.show()

print(f"Mean:   {y_311_train.mean():.1f} hours")
print(f"Median: {y_311_train.median():.1f} hours")

A mean well above the median is the signature of right skew: most complaints resolve quickly, but a long
tail of slow ones pulls the average up. Predicting a heavily skewed target directly is part of why the
residual plot above looks fan-shaped.

### A common fix: log-transform the target

Applying `log1p()` (log of $1+x$, so it stays defined at 0) compresses that long right tail, making the
target's distribution look a lot closer to symmetric — often a better match for what a straight line can
actually capture. "Often" is doing real work in that sentence, though — let's actually check, rather than
assume it helps here.

In [ ]:
y_311_train_log = np.__________(y_311_train)
y_311_test_log = np.__________(y_311_test)

model_311_log = LinearRegression()
model_311_log.fit(X_311_train_ready, y_311_train_log)
y_311_pred_log = model_311_log.predict(X_311_test_ready)

r2_311_log = r2_score(y_311_test_log, y_311_pred_log)
print(f"R² on raw hours:           {r2_311:.4f}")
print(f"R² on log-transformed target: {r2_311_log:.4f}")

However that comparison landed for you, resist reading too much into the direction. These two R² values
are measured on two different scales (hours vs. log-hours), so they're not strictly comparable in the
first place — and both are small enough that the difference between them is more about which scale
happens to fit the mechanics of least squares better than about which one gives you a genuinely better
real-world model. The honest lesson here isn't "log-transform always helps" or "log-transform didn't
help" — it's that a transform can reshape your target to better match what a linear model assumes, but
it cannot manufacture predictive signal that isn't in your features to begin with. `complaint_type`,
`borough`, and `hour_filed` were weak predictors before the transform, and they're weak predictors after
it. If you need predictions back in real hours, remember to apply `np.expm1()` to reverse the transform,
and re-check MAE/RMSE on that original scale before claiming any improvement.

### So why learn regression at all, if the answer is often "not much signal"?

Because knowing that, and being able to *show* it — cleanly, with the right metrics, and without
quietly hiding a bad number — is itself the valuable skill. Plenty of real data science work ends with
"these features don't explain much of this outcome," and that conclusion is only trustworthy if you
actually ran the correct process to get there: split honestly, fit correctly, evaluate with the right
metric, and check the residuals for what they can and can't tell you. Every one of those steps is exactly
what Parts 1 through 5 just walked through.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook:
`lect07_regression_exercise.ipynb`.

---

## Part 6 — Cheat Sheet

| Task | Code |
|---|---|
| Fit a linear regression | `LinearRegression().fit(X_train, y_train)` |
| Predict | `model.predict(X_test)` |
| Coefficients / intercept | `model.coef_`, `model.intercept_` |
| MAE | `mean_absolute_error(y_test, y_pred)` |
| MSE / RMSE | `mean_squared_error(y_test, y_pred)`, then `np.sqrt(...)` |
| R² | `r2_score(y_test, y_pred)` |
| Residuals | `y_test - y_pred` |
| Compress a right-skewed target | `np.log1p(y)`, reverse with `np.expm1(...)` |

---

## Part 7 — Key Terms

- **Simple linear regression**: predicting a continuous target from one feature, using a straight line.
- **Multiple linear regression**: predicting a continuous target from several features at once.
- **Residual**: the difference between an actual value and the model's prediction, $y_i - \hat{y}_i$.
- **Least squares**: the method of choosing model parameters that minimize the sum of squared residuals.
- **Normal equation**: the closed-form matrix solution for the least-squares coefficients,
  $\beta = (X^TX)^{-1}X^Ty$.
- **MAE / MSE / RMSE**: average error size, in original units (MAE, RMSE) or squared units (MSE); RMSE
  and MSE penalize large errors more than MAE does.
- **R² (coefficient of determination)**: the fraction of the target's variance the model explains,
  relative to always predicting the mean. Can be near zero, or even negative, on real data.
- **Residual plot**: a plot of residuals against predicted values, used to check whether a model's
  errors are patternless (good) or systematically patterned (a sign the model is missing something) —
  a different question from whether the model explains much of the outcome.
- **Heteroscedasticity**: when a model's errors have different spread across the range of predictions —
  a fan-shaped residual plot is the visual sign of it.
- **Right-skewed target**: a target with a long tail of unusually large values; mean well above median
  is a quick sign of it, and it can make squared-error metrics and a plain linear fit misleading.